In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.schema import Document

C:\Users\DNFH3173\AppData\Local\Temp\ipykernel_27100\3112106335.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [5]:
docs=[
    Document(page_content="Langchain is a powerful framework for building language model applications.", metadata={"source": "test1.txt"}),
    Document(page_content="FAISS is a library for efficient similarity search and clustering of dense vectors.", metadata={"source": "test2.txt"}),
    Document(page_content="BM25 is a ranking function used by search engines to rank matching documents according to their relevance to a given search query.", metadata={"source": "test3.txt"}),
    Document(page_content="Ensemble retrievers combine multiple retrieval methods to improve search results.", metadata={"source": "test4.txt"}),
    Document(page_content="Eiffel Tower is a iconic landmark in Paris, France.", metadata={"source": "test5.txt"})
]

In [6]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

c:\Users\DNFH3173\Desktop\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1698.10it/s]


In [7]:
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k=3

#combine with ensemble retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.5, 0.5]
    )





In [8]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000220576D4590>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000220576D4980>, k=3)], weights=[0.5, 0.5])

In [9]:
query = "What is Langchain?"
results = hybrid_retriever.invoke(query)


for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(f"Content: {doc.page_content}")
    print(f"Source: {doc.metadata['source']}")
    print()



Result 1:
Content: Langchain is a powerful framework for building language model applications.
Source: test1.txt

Result 2:
Content: FAISS is a library for efficient similarity search and clustering of dense vectors.
Source: test2.txt

Result 3:
Content: Eiffel Tower is a iconic landmark in Paris, France.
Source: test5.txt

Result 4:
Content: BM25 is a ranking function used by search engines to rank matching documents according to their relevance to a given search query.
Source: test3.txt

Result 5:
Content: Ensemble retrievers combine multiple retrieval methods to improve search results.
Source: test4.txt



### RAG Pipeline with hybrid retriever

In [13]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


In [10]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template("""
Answer the following question based on the context provided. If the answer is not contained within the context, say "I don't know."

Context:
{context}

Question:
{question}

""")

In [11]:

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("AZURE_FOUNDRY_MODEL"),
    base_url=os.getenv("AZURE_FOUNDRY_ENDPOINT"),
    api_key=os.getenv("AZURE_FOUNDRY_API_KEY"),
    temperature=0.7,
)

response = llm.invoke("What is the capital of France?")

print(response.content)


The capital of France is **Paris**.


In [ ]:
documents = create_stuff_documents_chain(llm=llm, prompt=prompt)

rag_chain = create_retrieval_chain(retriever=hybrid_retriever, combine_documents_chain=documents)

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7', 'langchain-openai': '1.7.3'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000220544E2350>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000220544E2AD0>, root_client=<openai.OpenAI object at 0x00000220544E1F90>, root_async_client=<openai.AsyncOpenAI object at 0x00000220544E2710>, model_name='Kimi-K2.7-Code', temperature=0.7, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://openhandsinfer.openai.azure.com/openai/v1', stream_chunk_timeout=120.0)